# Gold — Jornadas e etapas por origem

Jornada inferida = cliente + motivo + dia de abertura (America/Sao_Paulo).
URA no primeiro contato indica telefone; chatbot indica digital. Início humano
ou empate entre canais de origem fica indeterminado. Essa aproximação não comprova
transferências e separa contatos que atravessam a meia-noite.

## Entradas e execução

Executar após todas as tarefas Silver. Manter `jornadas.py` e `config.py` na mesma
pasta deste notebook. As tabelas atuais da Gold são preservadas.
As quatro novas tabelas são substituídas integralmente após as validações.
A publicação usa quatro gravações Delta independentes, sem transação entre tabelas.



In [ ]:
from pyspark.sql import functions as F
from config import SILVER_SCHEMA, GOLD_SCHEMA
from jornadas import construir_jornadas, validar_resultados

spark.conf.set('spark.sql.session.timeZone', 'America/Sao_Paulo')
chamados = spark.table(f'{SILVER_SCHEMA}.chamados')
horarios = spark.table(f'{SILVER_SCHEMA}.chamados_hora')
motivos = spark.table(f'{SILVER_SCHEMA}.base_motivos')
custos = spark.table(f'{SILVER_SCHEMA}.custos')
pesquisas = spark.table(f'{SILVER_SCHEMA}.pesquisa_satisfacao')
clientes = spark.table(f'{SILVER_SCHEMA}.clientes')
canais = spark.table(f'{SILVER_SCHEMA}.canais')


## Regras e indicadores

- Apenas aberturas entre 01/01/2025 e 31/07/2026 formam jornadas.
  Aberturas fora do período ficam nulas e são excluídas com motivo
  `abertura_fora_periodo`; o chamado permanece no vínculo para auditoria.

- `chamados.id_canal` é associado a `canais.nome_canal` dentro de `construir_jornadas`.
  O join preserva todos os chamados; canal ausente ou não cadastrado fica desconhecido.
  `tipo_canal` classifica humano/autosserviço e não identifica a etapa.

- O cliente de `chamados` define a jornada quando existe em `silver.clientes`.
  Cliente ausente ou não cadastrado permanece no vínculo, fora dos indicadores.
  O cliente existe somente em `chamados`; divergências da origem são
  contabilizadas durante a construção Silver de `chamados_hora`.

- O resultado final usa o status do último contato por abertura; empate conflitante
  ou status ausente permanece desconhecido.
- As etapas humanas são os canais atendimento inicial e atendimento especializado,
  e não a posição cronológica. Contatos repetidos no mesmo nível contam individualmente.
- Sucesso da etapa = contatos resolvidos / contatos com status conhecido.
- Custo médio do contato usa contatos com registros de custo inteiramente válidos.
- Custo acumulado inclui autosserviço e todos os contatos até o horário do contato
  atual, incluindo o bloco inteiro de horários empatados. Sua média usa apenas
  acumulados completos. Custos ausentes não viram zero.
- A duração observada vai da primeira abertura até o fim do último contato; não é
  uma medida comprovada de tempo até resolução. Unidade: segundos.
- Satisfação usa pesquisas individuais válidas; nota >= 4 é positiva.
- Taxas e proporções são frações entre 0 e 1. No Power BI, recompor médias e taxas
  pelas somas e denominadores, sem calcular média de médias.
- `total_jornadas` por etapa não é aditivo entre etapas: uma jornada pode passar
  por vários níveis. Para totais distintos entre etapas, usar `jornada_chamados`.

Comparações são descritivas: o especializado pode receber demandas mais difíceis.
A regra existente da Silver para custos negativos é herdada, não revalidada aqui.



In [ ]:
tabelas = construir_jornadas(chamados, horarios, motivos, custos, pesquisas, clientes, canais)
try:
    # Materializar e validar todos os resultados antes da primeira gravação.
    for nome, df in tabelas.items():
        df.persist()
        print(f'{nome}: {df.count()} registros')
    validar_resultados(chamados, tabelas)
    display(tabelas['jornada_chamados'].groupBy('motivo_exclusao').count())
    display(tabelas['jornada_chamados'].groupBy('cliente_cadastrado').count())
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}')
    for nome, df in tabelas.items():
        (df.write.format('delta').mode('overwrite')
           .option('overwriteSchema', 'true').saveAsTable(f'{GOLD_SCHEMA}.{nome}'))
finally:
    for df in tabelas.values():
        df.unpersist()


## Comparação dos níveis humanos

Resumo geral, ponderado por contato, de telefone e digital. A origem indeterminada
permanece disponível nas tabelas para acompanhar a cobertura da classificação.
Os denominadores mostram quantos registros sustentam cada resultado.



In [ ]:
etapas = spark.table(f'{GOLD_SCHEMA}.indicadores_etapas_jornada')
comparacao = (
    etapas.filter(F.col('origem').isin('telefone', 'digital'))
    .filter(F.col('etapa').isin('atendimento inicial', 'atendimento especializado'))
    .groupBy('origem', 'etapa')
    .agg(*[F.sum(c).alias(c) for c in [
        'total_contatos', 'contatos_resolvidos', 'contatos_status_conhecido',
        'contatos_status_desconhecido', 'soma_custo_contatos_completos',
        'contatos_custo_completo', 'soma_custo_acumulado_completo',
        'contatos_acumulado_completo',
    ]])
    .withColumn('taxa_sucesso', F.when(F.col('contatos_status_conhecido') > 0,
        F.col('contatos_resolvidos') / F.col('contatos_status_conhecido')))
    .withColumn('custo_medio_contato', F.when(F.col('contatos_custo_completo') > 0,
        F.col('soma_custo_contatos_completos') / F.col('contatos_custo_completo')))
    .withColumn('custo_medio_acumulado', F.when(F.col('contatos_acumulado_completo') > 0,
        F.col('soma_custo_acumulado_completo') / F.col('contatos_acumulado_completo')))
)
display(comparacao.orderBy('etapa', 'origem'))


## Validação no ambiente de destino

Executar todas as células no Databricks e conferir os totais e as exclusões.
O notebook é entregue sem resultados de execução do Workspace; testes locais com
casos sintéticos não substituem essa validação com as tabelas Silver reais.

